# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content item for one client on one reporting date, containing aggregated Search Console and Google Analytics metrics.
Time Window: June 2026 (month = '2026-06').

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


### Features
The following fields are safe to use as input features because they are historical metrics available before making a prediction:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`

### Label
The prediction target is:

- **Future organic clicks** (or another future performance metric derived after the prediction date).

### Context
These fields identify or organize the data but should not be used as model features:

- `report_date`
- `month`
- `client_hash_id`
- `content_hash_id`

### Excluded
The following fields are excluded because they are metadata or availability indicators rather than predictive signals:

- `client_has_gsc`
- `client_has_ga4`
- `gsc_data_available`
- `ga4_data_available`

**Reason:** These fields indicate whether data exists or identify records. They do not represent user behavior or content performance and could introduce bias or leakage if used as model features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily_sample']}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
"""

con.sql(query).df()

,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-13,client_1a730cb2640a1abf,content_8a5dd4ef03e333e8,2
1,2026-06-14,client_e00b29e582949543,content_762f369b413f1869,2
2,2026-06-19,client_e00b29e582949543,content_8e51f9165149270e,2
3,2026-06-19,client_b77d0d5f08f05e64,content_65a9bd83aebd0e58,2
4,2026-06-22,client_aef6ffea193da149,content_f919f4c31e610d44,2


In [24]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES['fact_daily_sample']};
"""

con.sql(query).df()

,total_rows,first_date,last_date
0,11694072,2026-06-01,2026-06-30


In [1]:
query = f"""
SELECT
    COUNT(*) AS march_rows,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM {TABLES['fact_daily_sample']}
WHERE month = '2026-06';
"""

con.sql(query).df()

NameError: name 'TABLES' is not defined

In [25]:
query = f"""
SELECT
    gsc_data_available,
    ga4_data_available,
    COUNT(*) AS rows
FROM {TABLES['fact_daily_sample']}
GROUP BY
    gsc_data_available,
    ga4_data_available;
"""

con.sql(query).df()

,gsc_data_available,ga4_data_available,rows
0,False,True,90510
1,True,True,554216
2,False,<NA>,1441169
3,True,False,2368462
4,False,False,6283456
5,True,<NA>,956259


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset contains historical aggregated Search Console and Google Analytics metrics only. It cannot explain external events, search intent, or algorithm changes. Data availability differs between clients, and duplicate records may exist for some client-content-date combinations. Only information available before the prediction time should be used to avoid data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.